In [3]:
import os
import gym
import math
import joblib
import mlflow
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from gym import spaces
import tensorflow as tf
from typing import Tuple
from typing import Optional
from collections import deque
from skopt import gp_minimize
import matplotlib.pyplot as plt
from skopt.space import Integer, Real
from skopt.utils import use_named_args
from datetime import datetime, timedelta
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
from pandas.tseries.frequencies import to_offset
from sklearn.model_selection import train_test_split
from tensorflow.keras import models, layers, optimizers
from tensorflow.keras.layers import Dense, Dropout, Input

In [18]:
path="./data"

In [19]:
dp = f"{path}/thesis-data.csv" # drl-dymy-thesis --> timestamp otherwise only date

# Load or Download Data
if os.path.exists(dp):
    data = pd.read_csv(dp, low_memory=False)

# Preprocess Data
if 'date' not in data.columns:
    data.reset_index(inplace=True)  # Reset index to bring 'date' as a column
# Convert column names to lowercase
data.columns = data.columns.str.lower()
# Ensure 'date' is the index
data["date"] = pd.to_datetime(data["date"])  # Convert to datetime format
data.set_index("date", inplace=True)
# data = data.drop("unnamed: 0", axis=1)

In [20]:
data.head(5)

,open,high,low,close,volume,ticker,unnamed: 7,unnamed: 8,unnamed: 9,unnamed: 10,...,unnamed: 39,unnamed: 40,unnamed: 41,unnamed: 42,unnamed: 43,unnamed: 44,unnamed: 45,unnamed: 46,unnamed: 47,unnamed: 48
date,,,,,,,,,,,,,,,,,,,,,
1965-01-05,1257.719971,1257.719971,1257.719971,1257.719971,0,N225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1965-01-06,1263.989990,1263.989990,1263.989990,1263.989990,0,N225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1965-01-07,1274.270020,1274.270020,1274.270020,1274.270020,0,N225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1965-01-08,1286.430054,1286.430054,1286.430054,1286.430054,0,N225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1965-01-12,1288.540039,1288.540039,1288.540039,1288.540039,0,N225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# --------------------------------------------------------------------------
# Helper Functions (very simplified placeholders for demonstration)
# --------------------------------------------------------------------------
def calculate_average(values):
    """Simple helper to compute average (mean)."""
    return np.mean(values)

def calculate_standard_deviation(values):
    """Simple helper to compute standard deviation."""
    return np.std(values, ddof=1)  # ddof=1 for sample std

def calculate_stochastic_rsi(values, period=14):
    """
    Placeholder Stochastic RSI calculation.
    Returns (k, d).
    Replace with your actual Stochastic RSI logic as needed.
    """
    if len(values) < period:
        return (np.nan, np.nan)
    min_val = np.min(values[-period:])
    max_val = np.max(values[-period:])
    if max_val - min_val == 0:
        k_value = 50.0
    else:
        k_value = (values[-1] - min_val) / (max_val - min_val) * 100
    d_value = k_value  # For demo, just set d = k
    return (k_value, d_value)

# --------------------------------------------------------------------------
# Main Class
# --------------------------------------------------------------------------
class StockIndicators:
    def __init__(self, df):
        """
        df must have columns: 'open', 'high', 'low', 'close', 'volume'
        """
        self.df = df.copy()
        self.opens = df['open'].values
        self.highs = df['high'].values
        self.lows = df['low'].values
        self.closes = df['close'].values
        self.volumes = df['volume'].values
        self.original_columns = set(self.df.columns)

    # ----------------------------------------------------------------
    # 1) Gaussian Channel
    # ----------------------------------------------------------------
    def gaussian_channel(self, period=20):
        """
        Calculates rolling mean, std, and upper_band for the closes.
        Also calculates and stores Stoch RSI (k, d) for reference.
        """
        mean_list = []
        std_list = []
        upper_list = []
        k_list = []
        d_list = []

        for i in range(len(self.closes)):
            if i < period:
                mean_list.append(np.nan)
                std_list.append(np.nan)
                upper_list.append(np.nan)
            else:
                window = self.closes[i-period:i]
                mean_val = calculate_average(window)
                std_val = calculate_standard_deviation(window)
                upper_band = mean_val + 2 * std_val
                mean_list.append(mean_val)
                std_list.append(std_val)
                upper_list.append(upper_band)

            # Also store Stochastic RSI at each bar
            k_val, d_val = calculate_stochastic_rsi(self.closes[:i+1])
            k_list.append(k_val)
            d_list.append(d_val)

        self.df['gaussian_mean'] = mean_list
        self.df['gaussian_std'] = std_list
        self.df['gaussian_upper'] = upper_list
        self.df['stoch_k'] = k_list
        self.df['stoch_d'] = d_list

    # ----------------------------------------------------------------
    # 2) MA Crossover
    # ----------------------------------------------------------------
    def ma_crossover(self, fast_period=10, slow_period=20):
        """
        Stores a rolling fast MA and slow MA.
        """
        fast_list = []
        slow_list = []

        for i in range(len(self.closes)):
            if i < slow_period:
                fast_list.append(np.nan)
                slow_list.append(np.nan)
            else:
                fast_ma = calculate_average(self.closes[i - fast_period : i])
                slow_ma = calculate_average(self.closes[i - slow_period : i])
                fast_list.append(fast_ma)
                slow_list.append(slow_ma)

        self.df['ma_fast'] = fast_list
        self.df['ma_slow'] = slow_list

    # ----------------------------------------------------------------
    # 3) Bollinger Bands
    # ----------------------------------------------------------------
    def bollinger_bands(self, period=20):
        """
        Stores Bollinger mean, std, upper band, lower band.
        """
        means = []
        stds = []
        ups = []
        downs = []

        for i in range(len(self.closes)):
            if i < period:
                means.append(np.nan)
                stds.append(np.nan)
                ups.append(np.nan)
                downs.append(np.nan)
            else:
                window = self.closes[i - period : i]
                mean_val = calculate_average(window)
                std_val = calculate_standard_deviation(window)
                up = mean_val + (2 * std_val)
                down = mean_val - (2 * std_val)
                means.append(mean_val)
                stds.append(std_val)
                ups.append(up)
                downs.append(down)

        self.df['boll_mean'] = means
        self.df['boll_std'] = stds
        self.df['boll_up'] = ups
        self.df['boll_down'] = downs

    # ----------------------------------------------------------------
    # 4) Parabolic SAR (very simplified, per snippet)
    # ----------------------------------------------------------------
    def parabolic_sar(self):
        """
        Example from snippet:
          parabolic SAR = min of all lows up to that point (if i>=2), else NaN.
        This is NOT the standard SAR formula, just a demonstration to match snippet.
        """
        sar_list = []
        for i in range(len(self.lows)):
            if i < 2:
                sar_list.append(np.nan)
            else:
                # the snippet does: sar = min(self.lows[:i])
                # We'll store that for each row
                sar_val = np.min(self.lows[:i])
                sar_list.append(sar_val)

        self.df['parabolic_sar'] = sar_list

    # ----------------------------------------------------------------
    # 5) Ichimoku Cloud (tenkan, kijun, senkou_a)
    # ----------------------------------------------------------------
    def ichimoku_cloud(self, period=26):
        tenkan_list = []
        kijun_list = []
        senkou_a_list = []

        for i in range(len(self.closes)):
            if i < period:
                tenkan_list.append(np.nan)
                kijun_list.append(np.nan)
                senkou_a_list.append(np.nan)
            else:
                # tenkan uses 9 bars
                tenkan_val = (np.max(self.highs[i-9:i]) + np.min(self.lows[i-9:i])) / 2
                kijun_val = (np.max(self.highs[i-period:i]) + np.min(self.lows[i-period:i])) / 2
                senkou_a_val = (tenkan_val + kijun_val) / 2
                tenkan_list.append(tenkan_val)
                kijun_list.append(kijun_val)
                senkou_a_list.append(senkou_a_val)

        self.df['ichimoku_tenkan'] = tenkan_list
        self.df['ichimoku_kijun'] = kijun_list
        self.df['ichimoku_senkou_a'] = senkou_a_list

    # ----------------------------------------------------------------
    # 6) MACD
    # ----------------------------------------------------------------
    def macd(self, fast=12, slow=26, signal=9):
        """
        We'll store macd_line and macd_signal as rolling averages
        (again, simplified approach).
        """
        macd_line_list = []
        signal_line_list = []

        for i in range(len(self.closes)):
            if i < slow:
                macd_line_list.append(np.nan)
                signal_line_list.append(np.nan)
            else:
                ema_fast = calculate_average(self.closes[i - fast : i])
                ema_slow = calculate_average(self.closes[i - slow : i])
                macd_val = ema_fast - ema_slow

                # For the signal line, we'll just do a (signal)-period average of closes
                # per the snippet logic
                if i < signal:
                    sig_val = np.nan
                else:
                    sig_val = calculate_average(self.closes[i - signal : i])

                macd_line_list.append(macd_val)
                signal_line_list.append(sig_val)

        self.df['macd_line'] = macd_line_list
        self.df['macd_signal'] = signal_line_list

    # ----------------------------------------------------------------
    # 7) RSI (based on snippet: using stoch RSI again)
    # ----------------------------------------------------------------
    def rsi(self, period=14):
        """
        We'll store a 'rsi_value' column, using the snippet's approach
        (which actually uses the K from Stochastic RSI).
        """
        rsi_vals = []
        for i in range(len(self.closes)):
            # Calculate stoch RSI
            k_val, _ = calculate_stochastic_rsi(self.closes[:i+1], period=period)
            rsi_vals.append(k_val)

        self.df['rsi_value'] = rsi_vals

    # ----------------------------------------------------------------
    # 8) ADX (based on snippet approach)
    # ----------------------------------------------------------------
    def adx(self, period=14):
        """
        Very simplified snippet-based ADX:
          up_moves = [high[j] - high[j-1]]
          down_moves = [low[j-1] - low[j]]
          adx = average of abs(up_moves - down_moves)
        for the last 'period' bars.
        """
        adx_list = []
        for i in range(len(self.closes)):
            if i < period:
                adx_list.append(np.nan)
            else:
                up_moves = []
                down_moves = []
                for j in range(i-period+1, i+1):
                    up_moves.append(self.highs[j] - self.highs[j-1])
                    down_moves.append(self.lows[j-1] - self.lows[j])
                # Pairwise difference
                diffs = [abs(u - d) for u, d in zip(up_moves, down_moves)]
                adx_val = calculate_average(diffs)
                adx_list.append(adx_val)

        self.df['adx_value'] = adx_list

    # ----------------------------------------------------------------
    # 9) ATR
    # ----------------------------------------------------------------
    def atr(self, period=14):
        """
        ATR per snippet:
          tr = max(high-low, abs(high-prev_close), abs(low-prev_close))
          atr = average(tr over period).
        """
        atr_list = []
        for i in range(len(self.closes)):
            if i < period:
                atr_list.append(np.nan)
            else:
                tr_values = []
                for j in range(i-period+1, i+1):
                    high_low = self.highs[j] - self.lows[j]
                    high_close = abs(self.highs[j] - self.closes[j-1])
                    low_close = abs(self.lows[j] - self.closes[j-1])
                    tr_values.append(max(high_low, high_close, low_close))
                atr_val = calculate_average(tr_values)
                atr_list.append(atr_val)

        self.df['atr_value'] = atr_list

    # ----------------------------------------------------------------
    # 10) VWMA
    # ----------------------------------------------------------------
    def vwma(self, period=20):
        """
        Volume Weighted Moving Average:
          price_vol = sum(closes * volume)
          vol_sum = sum(volume)
          vwma = price_vol / vol_sum
        """
        vwma_list = []
        for i in range(len(self.closes)):
            if i < period:
                vwma_list.append(np.nan)
            else:
                window_close = self.closes[i - period : i]
                window_vol = self.volumes[i - period : i]
                price_vol = np.sum(window_close * window_vol)
                vol_sum = np.sum(window_vol)
                if vol_sum == 0:
                    vwma_val = np.nan
                else:
                    vwma_val = price_vol / vol_sum
                vwma_list.append(vwma_val)

        self.df['vwma_value'] = vwma_list

    # ----------------------------------------------------------------
    # 11) Donchian Channel
    # ----------------------------------------------------------------
    def donchian_channel(self, period=20):
        """
        We'll just store the upper channel from the snippet.
        """
        upper_list = []
        for i in range(len(self.closes)):
            if i < period:
                upper_list.append(np.nan)
            else:
                upper_channel = np.max(self.highs[i - period : i])
                upper_list.append(upper_channel)

        self.df['donchian_upper'] = upper_list

    # ----------------------------------------------------------------
    # 12) Keltner Channel
    # ----------------------------------------------------------------
    def keltner_channel(self, period=20):
        """
        Snippet version:
          typical_price = (h + l + c)/3
          ema = average(typical_price)
          atr = average([h-l])
          upper_band = ema + 2*atr
        We'll store 'keltner_ema', 'keltner_atr', 'keltner_upper'.
        """
        ema_list = []
        atr_list = []
        upper_list = []

        for i in range(len(self.closes)):
            if i < period:
                ema_list.append(np.nan)
                atr_list.append(np.nan)
                upper_list.append(np.nan)
            else:
                tp_window = []
                hl_diff = []
                for j in range(i - period, i):
                    tp_window.append((self.highs[j] + self.lows[j] + self.closes[j]) / 3)
                    hl_diff.append(self.highs[j] - self.lows[j])

                ema_val = calculate_average(tp_window)
                atr_val = calculate_average(hl_diff)
                upper_band = ema_val + (2 * atr_val)

                ema_list.append(ema_val)
                atr_list.append(atr_val)
                upper_list.append(upper_band)

        self.df['keltner_ema'] = ema_list
        self.df['keltner_atr'] = atr_list
        self.df['keltner_upper'] = upper_list

    # ----------------------------------------------------------------
    # 13) Supertrend
    # ----------------------------------------------------------------
    def supertrend(self, period=10, multiplier=3):
        """
        Snippet logic:
          tr = [high - low for j in range(i-period, i)]
          atr = average(tr)
          basic_upper = (max(high[i-period:i]) + min(low[i-period:i]))/2 + (multiplier*atr)
        We'll store 'supertrend_basic_upper'.
        """
        st_upper_list = []
        for i in range(len(self.closes)):
            if i < period:
                st_upper_list.append(np.nan)
            else:
                tr_list = []
                for j in range(i - period, i):
                    tr_list.append(self.highs[j] - self.lows[j])
                atr_val = calculate_average(tr_list)
                highest = np.max(self.highs[i - period : i])
                lowest = np.min(self.lows[i - period : i])
                basic_upper = (highest + lowest) / 2 + (multiplier * atr_val)
                st_upper_list.append(basic_upper)

        self.df['supertrend_basic_upper'] = st_upper_list

    # ----------------------------------------------------------------
    # 14) Fibonacci
    # ----------------------------------------------------------------
    def fibonacci(self, period=20):
        """
        Snippet logic:
          high = max(self.highs[i-period:i])
          low = min(self.lows[i-period:i])
          fib_618 = low + (high - low)*0.618
        We'll store fib_618 in 'fib_618_value'.
        """
        fib_list = []
        for i in range(len(self.closes)):
            if i < period:
                fib_list.append(np.nan)
            else:
                highest = np.max(self.highs[i - period : i])
                lowest = np.min(self.lows[i - period : i])
                fib_618 = lowest + (highest - lowest)*0.618
                fib_list.append(fib_618)

        self.df['fib_618_value'] = fib_list

    # ----------------------------------------------------------------
    # 15) Price Action
    # ----------------------------------------------------------------
    def price_action(self):
        """
        Snippet logic:
          bullish_candle = closes[i] > opens[i].
        We'll store a boolean for each row: True/False if bullish.
        """
        bullish_list = (self.df['Close'] > self.df['Open']).tolist()
        self.df['priceaction_bullish'] = bullish_list

    # ----------------------------------------------------------------
    # 16) Volume Spike
    # ----------------------------------------------------------------
    def volume_spike(self, period=20):
        """
        Snippet logic:
          avg_volume (last period) = average(self.volumes[i-period : i-1])
          If self.volumes[i] > (2 * avg_volume) => spike.
        We'll store 'vol_spike_avg' and a boolean 'vol_spike'.
        """
        vol_avg_list = []
        vol_spike_list = []
        for i in range(len(self.volumes)):
            if i < period:
                vol_avg_list.append(np.nan)
                vol_spike_list.append(False)
            else:
                avg_vol = calculate_average(self.volumes[i - period : i])
                vol_avg_list.append(avg_vol)
                if self.volumes[i] > 2 * avg_vol:
                    vol_spike_list.append(True)
                else:
                    vol_spike_list.append(False)

        self.df['vol_spike_avg'] = vol_avg_list
        self.df['vol_spike'] = vol_spike_list

    # ----------------------------------------------------------------
    # 17) Momentum
    # ----------------------------------------------------------------
    def momentum(self, period=10):
        """
        Snippet logic:
          momentum = closes[i] - closes[i-period].
        We'll store 'momentum_value'.
        """
        mom_list = []
        for i in range(len(self.closes)):
            if i < period:
                mom_list.append(np.nan)
            else:
                mom_val = self.closes[i] - self.closes[i - period]
                mom_list.append(mom_val)

        self.df['momentum_value'] = mom_list

    # ----------------------------------------------------------------
    # 18) Williams %R
    # ----------------------------------------------------------------
    def williams_r(self, period=14):
        """
        Snippet logic:
          highest_high = max(self.highs[i-period:i])
          lowest_low = min(self.lows[i-period:i])
          wr = ((highest_high - close[i]) / (highest_high - lowest_low)) * -100
        We'll store 'williams_r_value'.
        """
        wr_list = []
        for i in range(len(self.closes)):
            if i < period:
                wr_list.append(np.nan)
            else:
                hh = np.max(self.highs[i - period : i])
                ll = np.min(self.lows[i - period : i])
                denom = (hh - ll)
                if denom == 0:
                    wr = np.nan
                else:
                    wr = ((hh - self.closes[i]) / denom) * -100
                wr_list.append(wr)

        self.df['williams_r_value'] = wr_list

    # ----------------------------------------------------------------
    # 19) CMF
    # ----------------------------------------------------------------
    def cmf(self, period=20):
        """
        Snippet logic:
          mfv[j] = (((close - low) - (high - close)) / (high - low)) * volume[j]
          cmf = sum(mfv) / sum(volumes) for the last 'period' bars.
        We'll store 'cmf_value'.
        """
        cmf_list = []
        for i in range(len(self.closes)):
            if i < period:
                cmf_list.append(np.nan)
            else:
                mfv_vals = []
                vol_vals = []
                for j in range(i - period, i):
                    high_low = self.highs[j] - self.lows[j]
                    if high_low == 0:
                        mfv = 0
                    else:
                        mfv = (((self.closes[j] - self.lows[j]) - (self.highs[j] - self.closes[j]))
                               / high_low) * self.volumes[j]
                    mfv_vals.append(mfv)
                    vol_vals.append(self.volumes[j])
                total_vol = np.sum(vol_vals)
                if total_vol == 0:
                    cmf_val = np.nan
                else:
                    cmf_val = np.sum(mfv_vals) / total_vol
                cmf_list.append(cmf_val)

        self.df['cmf_value'] = cmf_list

    # ----------------------------------------------------------------
    # 20) OBV
    # ----------------------------------------------------------------
    def obv(self):
        """
        Snippet logic:
          obv starts at 0, then for each j from 1..i:
            if close[j] > close[j-1]: obv += volume[j]
            elif close[j] < close[j-1]: obv -= volume[j]
        We'll store 'obv_value' for each row.
        """
        obv_values = []
        for i in range(len(self.closes)):
            if i == 0:
                obv_values.append(0)
            else:
                # compute OBV up to i
                obv_val = 0
                for j in range(1, i+1):
                    if self.closes[j] > self.closes[j-1]:
                        obv_val += self.volumes[j]
                    elif self.closes[j] < self.closes[j-1]:
                        obv_val -= self.volumes[j]
                obv_values.append(obv_val)
        self.df['obv_value'] = obv_values

    # ----------------------------------------------------------------
    # Method: run_all_indicators
    #   Dynamically calls every indicator method so you don't have to
    #   manually call them one by one.
    # ----------------------------------------------------------------
    def run_all_indicators(self):
        """
        Dynamically finds and calls all indicator methods (those that
        don't start with underscore or appear in the skip set).
        """
        skip_methods = {
            'run_all_indicators', 'clean_data', '__init__',
            '__module__', '__doc__', '__dict__', '__weakref__'
        }
        for attr_name in dir(self):
            if attr_name.startswith('_'):
                continue
            if attr_name in skip_methods:
                continue

            attr = getattr(self, attr_name)
            if callable(attr):
                print(f"Running: {attr_name}")
                attr()  # call the indicator method
        return self

    def get_new_indicator_columns(self):
        """
        Returns a list of columns that were added after the class
        was initialized—i.e. newly created indicator columns.
        """
        current_columns = set(self.df.columns)
        # The newly added columns are the difference
        new_columns = current_columns - self.original_columns
        return list(new_columns)

    # ----------------------------------------------------------------
    # Method: clean_data
    #   Example cleanup: drop rows with NaN in newly created columns.
    #   Also, optionally remove rows with zeros if needed.
    # ----------------------------------------------------------------
    def clean_data(self):
        """
        Drop rows that contain NaN in any newly created indicator columns.
        """
        # Original columns in your dataset:
        original_cols = {'open', 'high', 'low', 'close', 'volume'}
        new_cols = [c for c in self.df.columns if c not in original_cols]

        # Drop rows where any of these new columns has NaN:
        self.df.dropna(subset=new_cols, inplace=True)

        # Optionally drop rows with zeros in these columns:
        # mask = (self.df[new_cols] == 0).any(axis=1)
        # self.df = self.df[~mask]

        return self.df

In [27]:
class DataProcessor:
    """ Processes financial data for use in trading systems. """

    def __init__(self, data: pd.DataFrame, timeframe: Optional[str] = None):
        """
        Initialize the DataProcessor.

        Args:
            data (pd.DataFrame): Raw financial data.
            timeframe (Optional[str]): Timeframe for filtering data (e.g., '5YE').
        """
        self.raw_data = data.copy()
        self.timeframe = timeframe
        self._preprocess_data()
        self._validate_data()
        self._clean_data()

    def _preprocess_data(self):
        """Filter data based on the specified timeframe."""
        self.raw_data = self.raw_data.sort_index()

        if self.timeframe:
            offset = to_offset(self.timeframe)
            end_date = self.raw_data.index[-1]
            start_date = end_date - offset
            self.raw_data = self.raw_data.loc[start_date:end_date]

        self.raw_data = self.raw_data.ffill().bfill()

    def _validate_data(self):
        """Validate that required columns are present."""
        required = ['open', 'high', 'low', 'close', 'volume']
        missing = [col for col in required if col not in self.raw_data.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

    def _clean_data(self):
        """Clean and standardize the data."""
        self.raw_data.columns = self.raw_data.columns.str.lower()
        for col in ['timeframe', 'date']:
            if col in self.raw_data.columns:
                self.raw_data.drop(columns=[col], inplace=True)
        self.raw_data.dropna(inplace=True)
        self.raw_data.reset_index(drop=True, inplace=True)

    def add_technical_indicators(self):
        """Add technical indicators to the data."""
        """Calculate and add technical indicators"""
        df = self.raw_data.copy()
        ind_sys = StockIndicators(df)
        df = ind_sys.run_all_indicators()
        new_cols = ind_sys.get_new_indicator_columns()
        df = ind_sys.clean_data()
        self.raw_data = df
        return self.raw_data, new_cols


class DNNOptimizer:
    """Optimizes and trains a Deep Neural Network for financial data."""

    def __init__(self, input_dim: int):
        """
        Initialize the DNNOptimizer.

        Args:
            input_dim (int): Number of input features.
        """
        self.input_dim = input_dim
        self.scaler = MinMaxScaler()
        self.model = None

    def _build_model(self, num_layers: int = 3, units: int = 128, dropout: float = 0.2, lr: float = 0.001):
        """Build the DNN architecture."""
        model = models.Sequential()
        model.add(layers.Input(shape=(self.input_dim,)))
        for _ in range(num_layers):
            model.add(layers.Dense(units, activation='relu'))
            model.add(layers.Dropout(dropout))
        model.add(layers.Dense(1, activation='linear'))
        model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='mse', metrics=['mae'])
        return model

    def optimize_hyperparameters(self, X: np.ndarray, y: np.ndarray, n_calls: int = 15):
        """Optimize hyperparameters using Bayesian optimization."""
        dimensions = [
            Integer(2, 5, name='num_layers'),
            Integer(64, 256, name='units'),
            Real(0.1, 0.5, name='dropout'),
            Real(1e-4, 1e-2, name='lr')
        ]

        @use_named_args(dimensions)
        def objective(**params):
            """Objective function for optimization."""
            model = self._build_model(**params)
            X_scaled = self.scaler.fit_transform(X)
            X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)
            history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, verbose=0)
            return history.history['val_loss'][-1]

        result = gp_minimize(objective, dimensions, n_calls=n_calls, verbose=True)
        self.model = self._build_model(*result.x)
        self.model.fit(self.scaler.fit_transform(X), y, epochs=50, verbose=0)
        return result.x

    def save(self, path: str):
        """Save the model and scaler."""
        os.makedirs(path, exist_ok=True)
        self.model.save(os.path.join(path, 'dnn_model.keras'))
        joblib.dump(self.scaler, os.path.join(path, 'dnn_scaler.pkl'))

    @classmethod
    def load(cls, path: str):
        """Load a saved model and scaler."""
        model = models.load_model(os.path.join(path, 'dnn_model.keras'))
        input_dim = model.input_shape[1]
        dnn_optimizer = cls(input_dim)
        dnn_optimizer.model = model
        dnn_optimizer.scaler = joblib.load(os.path.join(path, 'dnn_scaler.pkl'))
        return dnn_optimizer



class TradingEnv(gym.Env):
    """A trading environment for reinforcement learning."""

    def __init__(self, data: pd.DataFrame, features: np.ndarray):
        """
        Initialize the TradingEnv.

        Args:
            data (pd.DataFrame): Financial data.
            features (np.ndarray): Feature matrix.
        """
        self.data = data
        self.features = features
        self.action_space = spaces.Discrete(3)  # 0: hold, 1: buy, 2: sell
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(features.shape[1] + 5 + 3,), dtype=np.float32)
        self.reset()

    def reset(self):
        """Reset the environment to its initial state."""
        self.current_step = 0
        self.balance = 100000.0
        self.holdings = 0
        self.trade_history = deque(maxlen=5)
        return self._next_observation()

    def _next_observation(self):
        """Generate the next observation."""
        market_data = self.data.iloc[self.current_step][['open', 'high', 'low', 'close', 'volume']].values.astype(np.float32)
        indicator_data = self.features[self.current_step].astype(np.float32)
        portfolio_state = np.array([self.holdings, self.balance, self.balance + self.holdings * self.data.iloc[self.current_step]['close']], dtype=np.float32)
        return np.concatenate([market_data, indicator_data, portfolio_state])

    def step(self, action: int):
        """Execute a trading action."""
        current_price = self.data.iloc[self.current_step]['close']
        prev_value = self.balance + self.holdings * current_price

        if action == 1 and self.balance > 10:  # Buy
            max_shares = self.balance // (current_price * 1.001)
            self.holdings += max_shares
            self.balance -= max_shares * current_price * 1.001
        elif action == 2 and self.holdings > 0:  # Sell
            self.balance += self.holdings * current_price * 0.999
            self.holdings = 0

        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        new_price = self.data.iloc[self.current_step]['close'] if not done else current_price
        reward = (self.balance + self.holdings * new_price) - prev_value
        return self._next_observation(), reward, done, {}


class DRLAgent:
    """A Deep Reinforcement Learning agent for trading."""

    def __init__(self, state_size: int, action_size: int):
        """
        Initialize the DRLAgent.

        Args:
            state_size (int): Size of the state space.
            action_size (int): Size of the action space.
        """
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target()

    def _build_model(self):
        """Build the DQN model."""
        model = models.Sequential([
            layers.Dense(64, activation='relu', input_dim=self.state_size),
            layers.BatchNormalization(),
            layers.Dense(32, activation='relu'),
            layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=optimizers.Adam(0.001))
        return model

    def update_target(self):
        """Update the target network weights."""
        self.target_model.set_weights(self.model.get_weights())

    def act(self, state: np.ndarray) -> int:
        """Select an action using epsilon-greedy policy."""
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        return np.argmax(self.model.predict(state[np.newaxis], verbose=0)[0])

    def remember(self, state: np.ndarray, action: int, reward: float, next_state: np.ndarray, done: bool):
        """Store experience in replay buffer."""
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size: int = 32):
        """Train the model using experience replay."""
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        states = np.array([t[0] for t in minibatch])
        next_states = np.array([t[3] for t in minibatch])

        targets = self.model.predict(states)
        next_q_values = self.target_model.predict(next_states)

        for i, (_, action, reward, _, done) in enumerate(minibatch):
            targets[i][action] = reward if done else reward + self.gamma * np.max(next_q_values[i])

        self.model.fit(states, targets, epochs=1, verbose=0)
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save_model(self, path: str):
        """Save the trained model."""
        os.makedirs(path, exist_ok=True)
        self.model.save(os.path.join(path, 'drl_model.keras'))

    @classmethod
    def load(cls, path: str):
        """Load a trained model."""
        model = models.load_model(os.path.join(path, 'drl_model.keras'))
        agent = cls(model.input_shape[1], model.output_shape[1])
        agent.model = model
        agent.update_target()
        return agent


class FinDRLSystem:
    """A complete trading system integrating DNN and DRL."""

    def __init__(self, data: pd.DataFrame, timeframe: Optional[str] = '10YE'):
        """
        Initialize the FinDRLSystem.

        Args:
            data (pd.DataFrame): Financial data.
            timeframe (Optional[str]): Timeframe for filtering data.
        """
        self._setup_mlflow()
        self.processor = DataProcessor(data, timeframe)
        self.data, indicator_cols = self.processor.add_technical_indicators()
        self.X = self.data[indicator_cols].values
        self.y = self.data['close'].pct_change().shift(-1).fillna(0).values
        self.prices = self.data['close'].values
        self.dnn_optimizer = DNNOptimizer(self.X.shape[1])
        self.dnn_optimizer.optimize_hyperparameters(self.X, self.y)
        self.features = self.dnn_optimizer.model.predict(self.dnn_optimizer.scaler.transform(self.X))
        self.env = TradingEnv(self.data, self.features)
        self.agent = DRLAgent(self.env.observation_space.shape[0], self.env.action_space.n)
        self.training_loss = []
        self.sharpe_ratios = []
        self.max_drawdowns = []
        self.episode_rewards = []
        self.episode_actions = []
        self.portfolio_history = []

    def _setup_mlflow(self):
        """Configure MLflow for tracking."""
        mlflow.set_tracking_uri("file://./data/models/mlruns")
        mlflow.set_experiment("FinDRL_Trading_Experiment")
        mlflow.tensorflow.autolog(log_input_examples=True, log_model_signatures=True, log_models=True, registered_model_name="FinDRL")

    def train(self, episodes: int = 100, batch_size: int = 128):
        """Train the DRL agent."""
        with mlflow.start_run():
            mlflow.log_params({
                "timeframe": self.processor.timeframe,
                "episodes": episodes,
                "batch_size": batch_size,
                "dnn_input_dim": self.dnn_optimizer.input_dim,
                "drl_state_size": self.agent.state_size,
                "drl_action_size": self.agent.action_size
            })
            for episode in range(episodes):
                state = self.env.reset()
                total_reward = 0
                episode_actions = []
                episode_portfolio = []

                while True:
                    current_value = self.env.balance + self.env.holdings * self.env.data.iloc[self.env.current_step]['close']
                    episode_portfolio.append(current_value)
                    action = self.agent.act(state)
                    episode_actions.append(action)
                    next_state, reward, done, _ = self.env.step(action)
                    self.agent.remember(state, action, reward, next_state, done)
                    state = next_state
                    total_reward += reward
                    if done:
                        final_value = self.env.balance + self.env.holdings * self.env.data.iloc[-1]['close']
                        episode_portfolio.append(final_value)
                        break

                self.episode_rewards.append(total_reward)
                self.episode_actions.append(episode_actions)
                self.portfolio_history.append(episode_portfolio)
                print(f"Episode {episode+1}/{episodes} | Reward: ${total_reward:.2f}")
                mlflow.log_metrics({
                    "episode_reward": total_reward,
                    "sharpe_ratio": self._calculate_sharpe_ratio(episode),
                    "max_drawdown": self._calculate_max_drawdown(episode),
                    "epsilon": self.agent.epsilon
                }, step=episode)

            self._log_mlflow_models()
            self._log_mlflow_artifacts()

    def _calculate_sharpe_ratio(self, episode_idx: int) -> float:
        """Calculate the Sharpe ratio for an episode."""
        returns = np.diff(self.portfolio_history[episode_idx]) / self.portfolio_history[episode_idx][:-1]
        if np.std(returns) == 0:
            return 0
        return np.sqrt(252) * np.mean(returns) / np.std(returns)

    def _calculate_max_drawdown(self, episode_idx: int) -> float:
        """Calculate the maximum drawdown for an episode."""
        portfolio = np.array(self.portfolio_history[episode_idx])
        peak = np.maximum.accumulate(portfolio)
        trough = portfolio[np.argmax(peak - portfolio)]
        return (peak[np.argmax(peak - portfolio)] - trough) / peak[np.argmax(peak - portfolio)]

    def _log_mlflow_models(self):
        """Log trained models to MLflow."""
        mlflow.tensorflow.log_model(self.dnn_optimizer.model, "dnn_model", registered_model_name="FinDRL-DNN")
        mlflow.tensorflow.log_model(self.agent.model, "drl_agent", registered_model_name="FinDRL-Agent")

    def _log_mlflow_artifacts(self):
        """Save and log performance artifacts."""
        artifact_path = "./data/models/artifacts"
        os.makedirs(artifact_path, exist_ok=True)
        self.plot_full_analysis()
        plt.savefig(f"{artifact_path}/full_analysis.png")
        plt.close()
        self.plot_model_performance()
        plt.savefig(f"{artifact_path}/model_performance.png")
        plt.close()
        mlflow.log_artifacts(artifact_path, "performance_plots")
        shutil.rmtree(artifact_path)

    def plot_max_drawdown(self, episode_idx: int = -1):
        """Plot the drawdown curve for a specific episode."""
        portfolio = self.portfolio_history[episode_idx]
        rolling_max = np.maximum.accumulate(portfolio)
        drawdown = (rolling_max - portfolio) / rolling_max

        plt.figure(figsize=(12, 6))
        plt.fill_between(range(len(drawdown)), drawdown * 100, color='red', alpha=0.3)
        plt.plot(drawdown * 100, color='darkred')
        plt.title(f"Drawdown Curve (Episode {episode_idx+1})")
        plt.ylabel("Drawdown (%)")
        plt.xlabel("Time Step")
        plt.gca().yaxis.set_major_formatter(PercentFormatter())
        plt.grid(True)
        plt.show()

    def plot_sharpe_ratio(self):
        """Plot the evolution of Sharpe ratios across episodes."""
        self.sharpe_ratios = [self._calculate_sharpe_ratio(i) for i in range(len(self.portfolio_history))]
        plt.figure(figsize=(12, 6))
        plt.plot(self.sharpe_ratios, marker='o', linestyle='-', color='blue')
        plt.title("Sharpe Ratio Evolution")
        plt.xlabel("Episode")
        plt.ylabel("Annualized Sharpe Ratio")
        plt.grid(True)
        plt.show()

    def plot_model_performance(self):
        """Visualize key performance metrics."""
        if not self.portfolio_history:
            print("No training data available!")
            return

        sharpe_ratios = []
        max_drawdowns = []
        annual_returns = []

        for portfolio in self.portfolio_history:
            returns = np.diff(portfolio) / portfolio[:-1]
            sharpe = np.sqrt(252) * np.mean(returns) / np.std(returns) if len(returns) > 0 and np.std(returns) > 0 else 0
            sharpe_ratios.append(sharpe)

            peak = np.maximum.accumulate(portfolio)
            trough = portfolio[np.argmax(peak - portfolio)]
            drawdown = (peak[np.argmax(peak - portfolio)] - trough) / peak[np.argmax(peak - portfolio)]
            max_drawdowns.append(drawdown)

            annual_returns.append((portfolio[-1]/portfolio[0])**(252/len(portfolio)) - 1)

        fig, axs = plt.subplots(2, 2, figsize=(16, 12))

        # Sharpe Ratio Evolution
        axs[0, 0].plot(sharpe_ratios, color='blue', marker='o')
        axs[0, 0].set_title('Sharpe Ratio Evolution')
        axs[0, 0].set_ylabel('Annualized Sharpe Ratio')
        axs[0, 0].grid(True)

        # Max Drawdown Progression
        axs[0, 1].plot(np.array(max_drawdowns) * 100, color='red', marker='s')
        axs[0, 1].set_title('Maximum Drawdown')
        axs[0, 1].set_ylabel('Drawdown (%)')
        axs[0, 1].grid(True)

        # Annualized Returns
        axs[1, 0].plot(np.array(annual_returns) * 100, color='green', marker='^')
        axs[1, 0].set_title('Annualized Returns')
        axs[1, 0].set_ylabel('Return (%)')
        axs[1, 0].grid(True)

        # Risk-Return Profile
        axs[1, 1].scatter(np.array(max_drawdowns) * 100, np.array(annual_returns) * 100, c=sharpe_ratios, cmap='viridis')
        axs[1, 1].set_title('Risk-Return Profile')
        axs[1, 1].set_xlabel('Max Drawdown (%)')
        axs[1, 1].set_ylabel('Annual Return (%)')
        axs[1, 1].grid(True)
        plt.colorbar(axs[1, 1].collections[0], ax=axs[1, 1], label='Sharpe Ratio')

        plt.tight_layout()
        plt.show()

    def plot_full_analysis(self, episode_idx: int = -1):
        """Comprehensive performance dashboard."""
        plt.figure(figsize=(18, 15))

        # Trading Signals
        plt.subplot(3, 2, 1)
        self._plot_trading_actions(episode_idx)

        # Learning Curve
        plt.subplot(3, 2, 2)
        self.plot_learning_curve()

        # Max Drawdown
        plt.subplot(3, 2, 3)
        self.plot_max_drawdown(episode_idx)

        # Sharpe Ratio
        plt.subplot(3, 2, 4)
        self.plot_sharpe_ratio()

        # Model Performance
        plt.subplot(3, 2, 5)
        self.plot_model_performance()

        # Portfolio Growth
        plt.subplot(3, 2, 6)
        self._plot_portfolio_growth(episode_idx)

        plt.tight_layout()
        plt.show()

    def plot_learning_curve(self):
        """Plot training performance over episodes."""
        plt.figure(figsize=(10, 5))
        plt.plot(self.episode_rewards)
        plt.title("Training Performance")
        plt.xlabel("Episode")
        plt.ylabel("Total Reward")
        plt.grid(True)
        plt.show()

    def plot_trading_actions(self, episode_idx: int = -1):
        """Visualize trading signals for a specific episode."""
        if not self.episode_actions:
            print("No trading actions recorded!")
            return

        actions = self.episode_actions[episode_idx]
        valid_length = min(len(actions), len(self.prices) - 1)
        buy_signals = [i for i, a in enumerate(actions[:valid_length]) if a == 1]
        sell_signals = [i for i, a in enumerate(actions[:valid_length]) if a == 2]

        plt.figure(figsize=(15, 5))
        plt.plot(self.prices[:valid_length + 1], label='Price')

        if isinstance(self.data.index, pd.DatetimeIndex):
            plt.scatter(self.data.index[buy_signals], self.prices[buy_signals], color='green', marker='^', label='Buy', s=100)
            plt.scatter(self.data.index[sell_signals], self.prices[sell_signals], color='red', marker='v', label='Sell', s=100)
        else:
            plt.scatter(buy_signals, self.prices[buy_signals], color='green', marker='^', label='Buy', s=100)
            plt.scatter(sell_signals, self.prices[sell_signals], color='red', marker='v', label='Sell', s=100)

        plt.title(f"Trading Signals - Episode {episode_idx + 1}")
        plt.xlabel("Date" if isinstance(self.data.index, pd.DatetimeIndex) else "Step")
        plt.ylabel("Price")
        plt.legend()
        plt.grid(True)
        plt.show()

    def _plot_trading_actions(self, episode_idx: int = -1, ax=None):
        """Internal method to plot price and trading actions."""
        if not self.episode_actions:
            print("No trading actions recorded!")
            return

        actions = self.episode_actions[episode_idx]
        valid_length = min(len(actions), len(self.prices) - 1)
        buy_signals = [i for i, a in enumerate(actions[:valid_length]) if a == 1]
        sell_signals = [i for i, a in enumerate(actions[:valid_length]) if a == 2]

        ax = ax or plt.gca()
        ax.plot(self.prices[:valid_length + 1], label='Price', alpha=0.8)
        ax.scatter(buy_signals, self.prices[buy_signals], color='green', marker='^', label='Buy', s=100)
        ax.scatter(sell_signals, self.prices[sell_signals], color='red', marker='v', label='Sell', s=100)
        ax.set_title(f"Trading Signals - Episode {episode_idx + 1}")
        ax.legend()
        ax.grid(True)

    def _plot_portfolio_growth(self, episode_idx: int = -1):
        """Plot portfolio value over time."""
        portfolio = self.portfolio_history[episode_idx]
        plt.plot(portfolio, label='Portfolio Value', color='purple')
        plt.title("Portfolio Growth")
        plt.xlabel("Time Step")
        plt.ylabel("Value ($)")
        plt.grid(True)
        plt.legend()

    def save_models(self, directory: str):
        """Save all system components."""
        os.makedirs(directory, exist_ok=True)
        self.dnn_optimizer.save(os.path.join(directory, 'dnn'))
        self.agent.save_model(os.path.join(directory, 'drl'))

    @classmethod
    def load(cls, new_data: pd.DataFrame, directory: str, timeframe: Optional[str] = '5YE'):
        """Load trained system with new data."""
        processor = DataProcessor(new_data, timeframe)
        data_with_indicators = processor.add_technical_indicators()[0]
        dnn_optimizer = DNNOptimizer.load(os.path.join(directory, 'dnn'))

        try:
            X = data_with_indicators[dnn_optimizer.scaler.feature_names_in_]
        except KeyError as e:
            raise ValueError(f"Feature mismatch in new data: {e}")

        features = dnn_optimizer.model.predict(dnn_optimizer.scaler.transform(X))
        system = cls.__new__(cls)
        system.__dict__.update({
            'processor': processor,
            'data': data_with_indicators,
            'dnn_optimizer': dnn_optimizer,
            'env': TradingEnv(data_with_indicators, features),
            'agent': DRLAgent.load(os.path.join(directory, 'drl')),
            'prices': data_with_indicators['close'].values,
            'episode_rewards': [],
            'episode_actions': []
        })
        return system

    @classmethod
    def load_production_model(cls, data: pd.DataFrame):
        """Load production models from MLflow."""
        mlflow.set_tracking_uri("file://./data/models/mlruns")
        dnn_model = mlflow.tensorflow.load_model("models:/FinDRL-DNN/Production")
        drl_model = mlflow.tensorflow.load_model("models:/FinDRL-Agent/Production")

        system = cls.__new__(cls)
        system.dnn_optimizer = DNNOptimizer(dnn_model.input_shape[1])
        system.dnn_optimizer.model = dnn_model
        system.agent = DRLAgent(drl_model.input_shape[1], drl_model.output_shape[1])
        system.agent.model = drl_model
        system._post_load_setup(data)
        return system

    def _post_load_setup(self, data: pd.DataFrame):
        """Reinitialize system components after loading."""
        self.processor = DataProcessor(data)
        self.data, indicator_cols = self.processor.add_technical_indicators()
        self.X = self.data[indicator_cols].values
        self.features = self.dnn_optimizer.model.predict(self.dnn_optimizer.scaler.transform(self.X))
        self.env = TradingEnv(self.data, self.features)
        self.prices = self.data['close'].values
        self.portfolio_history = []

    def save_models(self):
        """Save models to local directory (MLflow compatible)."""
        super().save_models("./data/models/native")
        self._log_mlflow_models()
        self._log_mlflow_artifacts()

In [28]:
system = FinDRLSystem(data, timeframe='5YE')
system.train(episodes=100, batch_size=128, )

OSError: [Errno 30] Read-only file system: '/data'